# TFM DASHBOARD

In [1]:
import pandas as pd

df_eda = pd.read_csv(r"..\00_Data\00_Processed\df_eda.csv")

In [2]:
df_eda['race'] = df_eda['race'].replace({'asian': 'other', 'native american': 'other'})

In [3]:
df_eda.to_csv(r'..\00_Data\00_Processed\df_dashboard.csv', index=False)

In [4]:
df_eval = pd.read_csv(r"..\00_Data\00_Processed\df_eval.csv")

In [5]:
df_eval['sex'] = df_eval['sex'].replace({0: 'male', 1: 'female'})

In [6]:
df_eval_dashboard = df_eval[[
    'person_id','y_true','y_score','y_pred','race', 'sex', 'age_cat'
]]

df_eval_dashboard.to_csv(r'..\00_Data\00_Processed\df_eval_dashboard.csv', index=False)

In [7]:
df_sin_outliers = pd.read_csv(r"..\00_Data\00_Processed\df_sin_outliers.csv")

In [8]:
df_eval = df_eval.merge(
    df_sin_outliers[['person_id','decile_score']],
    on='person_id',
    how='left'
)

df_eval['score_propublica'] = df_eval['decile_score'] / 10

In [9]:
df_eval.head()

,person_id,y_true,race,sex,age_cat,y_score,y_pred,decile_score,score_propublica
0,59474.0,0,other,male,25-45,0.435356,0,1,0.1
1,59890.0,1,african-american,male,less than 25,0.489725,0,3,0.3
2,61156.0,0,hispanic,male,less than 25,0.534697,1,1,0.1
3,57039.0,0,caucasian,female,25-45,0.360833,0,2,0.2
4,58070.0,1,other,male,less than 25,0.596857,1,4,0.4


In [10]:
df_eval['decil_modelo'] = pd.qcut(df_eval['y_score'], 10, labels=False, duplicates='drop')
df_eval['decil_propublica'] = pd.qcut(df_eval['score_propublica'], 10, labels=False, duplicates='drop')

# invertir → 1 = alto riesgo
df_eval['decil_modelo'] = df_eval['decil_modelo'].max() - df_eval['decil_modelo'] + 1
df_eval['decil_propublica'] = df_eval['decil_propublica'].max() - df_eval['decil_propublica'] + 1

In [24]:
df_eval_long = pd.concat([

    # TU MODELO
    df_eval[['person_id','race','sex','age_cat','y_true','decil_modelo']]
        .rename(columns={'decil_modelo':'decil'})
        .assign(modelo='mi_modelo'),

    # PROPUBLICA
    df_eval[['person_id','race','sex','age_cat','y_true','decil_propublica']]
        .rename(columns={'decil_propublica':'decil'})
        .assign(modelo='propublica')

])

In [ ]:
# df_lift = pd.concat([
#     df_eval.groupby(['decil_modelo', 'race', 'sex', 'age_cat'])['y_true'].mean().reset_index()
#         .rename(columns={'decil_modelo':'decil','y_true':'recid_rate'})
#         .assign(modelo='propuesta_modelo'),

#     df_eval.groupby(['decil_propublica', 'race', 'sex', 'age_cat'])['y_true'].mean().reset_index()
#         .rename(columns={'decil_propublica':'decil','y_true':'recid_rate'})
#         .assign(modelo='propublica')
# ])

In [17]:
df_fairness = df_eval.groupby(['race', 'sex', 'age_cat']).apply(
    lambda g: pd.Series({
        'FPR': ((g['y_pred']==1)&(g['y_true']==0)).sum() / ((g['y_true']==0).sum()),
        'TPR': ((g['y_pred']==1)&(g['y_true']==1)).sum() / ((g['y_true']==1).sum())
    })
).reset_index()

C:\Users\JAIME\AppData\Local\Temp\ipykernel_110956\4188917521.py:4: RuntimeWarning: invalid value encountered in scalar divide
  'TPR': ((g['y_pred']==1)&(g['y_true']==1)).sum() / ((g['y_true']==1).sum())
C:\Users\JAIME\AppData\Local\Temp\ipykernel_110956\4188917521.py:3: RuntimeWarning: invalid value encountered in scalar divide
  'FPR': ((g['y_pred']==1)&(g['y_true']==0)).sum() / ((g['y_true']==0).sum()),
C:\Users\JAIME\AppData\Local\Temp\ipykernel_110956\4188917521.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_fairness = df_eval.groupby(['race', 'sex', 'age_cat']).apply(


In [27]:
df_eval_long.to_csv(r'..\00_Data\00_Processed\df_eval_long.csv', index=False)
#df_lift.to_csv(r'..\00_Data\00_Processed\df_lift.csv', index=False)
df_fairness.to_csv(r'..\00_Data\00_Processed\df_fairness.csv', index=False)